# Member C -- Exploration (Epsilon) Experiments

Baseline config comes from `shared_train.py`. Only the exploration
params (`exploration_initial_eps`, `exploration_final_eps`,
`exploration_fraction`) vary across these 10 combos -- everything else
must stay at the shared baseline so results are comparable across the
whole group.

In [ ]:
!pip install -q "stable-baselines3[extra]" "gymnasium[atari,accept-rom-license]" ale-py "autorom[accept-rom-license]" pandas
!AutoROM --accept-license

## Get shared_train.py

Clone the group repo (or upload `shared_train.py` to this Colab session)
so the import below works.

In [ ]:
import os

REPO_DIR = "/content/deep-q-learning-formative3"
if os.path.exists(REPO_DIR):
    # reset --hard (not pull) so this always matches the remote exactly,
    # even if history upstream was rewritten (force-pushed) since your
    # last clone -- a plain pull can't reconcile that.
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} reset --hard origin/main
else:
    !git clone https://github.com/Mikekimm/deep-q-learning-formative3.git {REPO_DIR}
%cd {REPO_DIR}

from shared_train import BASELINE_CONFIG, GAME_ID, TOTAL_TIMESTEPS, SEED, train_one_run
print(GAME_ID, TOTAL_TIMESTEPS, SEED)
print(BASELINE_CONFIG)

## Smoke test -- run this ONE cell before anything below

This runs the exact same pipeline as a real experiment, but for only
2,000 timesteps (~1-2 minutes) instead of 200,000 (~hours). It exists
to catch install/setup errors early in *your* session -- Colab sessions
can have their own quirks independent of the code. Do NOT use "Run All"
on this notebook -- run this cell by itself first, confirm it prints a
result with no errors, THEN move on to the real experiment cells below.

In [ ]:
import shared_train

_original_total_timesteps = shared_train.TOTAL_TIMESTEPS
shared_train.TOTAL_TIMESTEPS = 2000  # temporary override for this test only

smoke_row = shared_train.train_one_run(overrides={}, run_name="smoketest", member="test")
print(smoke_row)

shared_train.TOTAL_TIMESTEPS = _original_total_timesteps  # restore before real runs below
print("TOTAL_TIMESTEPS restored to", shared_train.TOTAL_TIMESTEPS)

In [ ]:
# 10 combos of (initial_eps, final_eps, fraction of training spent decaying).
# Each changes one dimension of exploration relative to the baseline so the
# effect stays interpretable.
epsilon_combos = [
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.20, "exploration_fraction": 0.05},
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.10, "exploration_fraction": 0.05},
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.05, "exploration_fraction": 0.05},
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.05, "exploration_fraction": 0.10},
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.05, "exploration_fraction": 0.20},
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.05, "exploration_fraction": 0.30},
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.01, "exploration_fraction": 0.10},
    {"exploration_initial_eps": 0.8, "exploration_final_eps": 0.05, "exploration_fraction": 0.10},
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.30, "exploration_fraction": 0.10},
    {"exploration_initial_eps": 1.0, "exploration_final_eps": 0.05, "exploration_fraction": 0.50},
]

## Optional: watch reward curves live while training runs

Run this once before starting the loop below to open an inline
TensorBoard dashboard -- it updates live as each run trains, so you can
watch reward trends without waiting for a run to finish. (This shows the
reward *graph*, not the game itself -- see the README for why we don't
render pixels during training.)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir results/tb_logs

In [ ]:
results = []
for i, combo in enumerate(epsilon_combos, start=1):
    row = train_one_run(
        overrides=combo,
        run_name=f"memberC_eps_{i:02d}",
        member="MemberC",
    )
    results.append(row)

## Notes

After each run, record what you observed -- did slower decay (larger
fraction) improve final performance at the cost of slower early
learning? Did a high final_eps prevent convergence by never settling
into exploitation? Explain *why*, not just what.